# Local storage example

This notebook demonstrates `exa.core.local_storage.LocalStorage`, a fully local, file-backed
implementation of `StorageInterface` that requires no database or server. It shows how to:

- run an experiment using calibration loaded from a live backend,
- save the resulting observations both to the live backend and to a local file,
- load observations from local storage alongside a live calibration set,
- build a new observation set purely in local storage, and
- reload that local observation set later.

This is useful for e.g. keeping scratch/exploratory observations out of the shared database, or working
fully offline against previously saved data.

In [ ]:
%matplotlib inline

In [ ]:
import exa.core as ec
from exa.core.provider import provider
from exa.core.local_storage import LocalStorage
from exa.public_experiments import Rabi
from iqm.station_control.interface.models import ObservationSetDefinition, ObservationSetType

## Setup

- Connect to a station
- Use a matching experiment configuration file

In [ ]:
provider.init_iqm_server_runtime(
    iqm_server_url="IQM_SERVER_URL",
    # experiment_configuration_path="../../../exa-core/docs/notebooks/experiment.yml", # optional
    username="USERNAME",
    quantum_computer="qc_name",
    token="token_here, or alternatively set the IQM_TOKEN env varible"
)

## Initialize local storage

`LocalStorage` can be used purely in-memory (`LocalStorage()`), or pointed at a JSON file so its
contents survive across notebook/kernel restarts. We use the latter here.

In [ ]:
local_storage = LocalStorage("local_storage_example.json")

## Load the live default calibration set

`GetCalibrationSet("default")` loads the backend's current default calibration set (this replaces the
now-deprecated `GetDefaultCalibrationSet` rule). Registering the resulting stash as the global default
observation loading rules makes it the calibration source for the experiments run below.

In [ ]:
rules = [ec.GetCalibrationSet("default")]
obs_stash = ec.get_loaded_stash_from_rules(rules)
ec.set_global_default_observation_loading_rules([ec.LatestFromStash(obs_stash)])
tags = ["local_storage_example"]

## Run Rabi

The simplest way to execute and analyze an experiment, relying on the calibration loaded above.

We do this as an easy way to generate some observations.

In [ ]:
result = Rabi().run().analyze()
result.render_report()

## Save the results, to both the live backend and local storage

Each `save()` call independently persists the tagged observations: the first call writes to the live
backend (the ambient storage configured above), the second writes the same observations into
`local_storage` instead, via the `storage_interface` override. These end up as two separate sets of
observations, one per backend, sharing the same tag.

In [ ]:
result.observations.validate().tag(tags=tags).stash(obs_stash)
obs_stash.save(storage=local_storage)

## Load a second stash: local tag + live default calibration

Loading rules can mix storages freely: here `GetCalibrationSet("default")` still reads from the live
backend, while `GetLatestForTags(tags, storage=local_storage)` reads only from
`local_storage`, in the same call.

In [ ]:
rules_2 = [ec.GetCalibrationSet("default"), ec.GetLatestForTags(tags, storage=local_storage)]
obs_stash_2 = ec.get_loaded_stash_from_rules(rules_2)
obs_stash_2.get_db_observations()

## Build an observation set, in local storage

`exa.core.observation_set_utils.create_observation_set` only targets the ambient storage, so to create
an observation set in `local_storage` specifically, we call its `create_observation_set` method
directly with a hand-built `ObservationSetDefinition`.

Note that all observations you add to the set must already exist in `local_storage`

In [ ]:
# Get our previously tagged observations
rules_3 = [ec.GetLatestForTags(tags, storage=local_storage)]
obs_stash_3 = ec.get_loaded_stash_from_rules(rules_3)

In [ ]:
observations = [obs for obs_list in obs_stash_3.get_db_observations().values() for obs in obs_list]
local_observation_set = local_storage.create_observation_set(
    ObservationSetDefinition(
        observation_set_type=ObservationSetType.GENERIC_SET,
        observation_ids=[obs.observation_id for obs in observations],
    )
)
local_observation_set.observation_set_id

## Load the new local observation set

Finally, reload the observation set we just created, entirely from local storage.

In [ ]:
rules_4 = [ec.GetObservationSet(local_observation_set.observation_set_id, storage=local_storage)]
obs_stash_4 = ec.get_loaded_stash_from_rules(rules_4)
obs_stash_4.get_db_observations()